# PlanPtycho

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/electronmicroscopy/quantem.widget/blob/main/docs/tutorials/planptycho.ipynb)

`PlanPtycho` checks multislice ptychography settings against a known crystal before the experiment. Give it a crystal
(a CIF file, an `ase.Atoms`, or a Materials Project id) and the microscope settings; it shows where the beam goes
through the specimen, what the reconstruction's model window holds, what lands on the detector, and a list of checks
that say what fits and what to change.

In [ ]:
# @title Install QuantEM { display-mode: "form" }
if get_ipython().__class__.__module__.startswith("google.colab"):
    !wget -q https://raw.githubusercontent.com/electronmicroscopy/quantem.widget/main/scripts/install_colab.py -O install_quantem.py
    %run install_quantem.py
    !pip install -q abtem

In [ ]:
from ase import Atoms

# cubic perovskite SrTiO3, Pm-3m, a = 3.905 A
srtio3 = Atoms(
    "SrTiO3",
    scaled_positions=[(0, 0, 0), (0.5, 0.5, 0.5), (0.5, 0.5, 0), (0.5, 0, 0.5), (0, 0.5, 0.5)],
    cell=[3.905, 3.905, 3.905],
    pbc=True,
)

## Plan a thick, tilted crystal

110 nm of SrTiO3 viewed along [001], tilted (3.2, -2.4) mrad off the zone axis, with the default microscope preset: 300 kV,
a 30 mrad probe and the Arina (192 x 192 px) at a 91 mm camera length, 0.554 mrad per pixel. The probe is focused 17 nm
below the entrance surface and the scan is 48 x 48 positions with 0.495 A steps.

- **Top**: the projected crystal, smeared by the tilt across the thickness, the scan field (cyan), the probe at the scan
  centre with its beam at the view depth (yellow) and the reconstruction's model window (dashed). Drag to move the probe.
- **Side**: the specimen as a stationary slab. Drag the dashed focus line to move the focus; the labels show how much of
  the specimen lies above and below it. Drag elsewhere to set the view depth.
- **Probe**: the probe on the model window at the view depth, over the crystal at that depth (**Sample** switch). A beam
  wider than the window wraps around, as it does in the reconstruction.
- **Detector**: the bright-field disk, the Bragg disks, the zone axis moved by the tilt, and the detector edge.

Every slider updates all four panels and the checks. The wheel zooms a panel; double-click resets it. The menus in the
title bar apply a microscope preset or the recommended settings for 20-70 nm.

In [ ]:
from quantem.widget import PlanPtycho

plan = PlanPtycho(srtio3, thickness_nm=110, tilt_mrad=(3.2, -2.4), c10_nm=-17, scan_step_A=0.495, scan_size_px=48)
plan

## The checks as a table

`report()` returns the same checks as a DataFrame, for scripts and lab notebooks. Here the 57 A beam at the exit surface is
wider than the 35 A model window, the 24 A scan is too small for the deep beam, and the columns lean 4.4 A across the
thickness: all three are cautions, each with the evidence and what to change.

In [ ]:
plan.report()

## A collaborator's acquisition, presets, your own crystal

To check whether settings someone else recorded can work for ptychography, pass them as reported. A sampling without a
camera name makes the camera `custom`, and `c10_nm` is the defocus in the quantem sign (negative focuses into the
specimen). In the widget, click any number to type a value.

```python
PlanPtycho(srtio3, thickness_nm=40, voltage_kV=200, semiangle_mrad=24.5,
           detector_px=128, detector_mrad_per_px=0.9, scan_step_A=0.4, scan_size_px=256, c10_nm=-15)
```

`apply_thickness` sets the recommended focus and scan size for 20-200 nm, and `apply_preset` switches the microscope
settings. For cryo and biological sections of 150-200 nm, the beam at 30 mrad outgrows the Arina's 35.5 A window at
91 mm; a longer camera length widens the window but lowers the detector reach, and a smaller semiangle narrows the beam.

```python
plan.apply_thickness(200)
plan.apply_preset("Arina · 300 kV · 21.4 mrad · 185 mm")
PlanPtycho("my_crystal.cif", zone_axis=(0, 1, 1), thickness_nm=40, preset="Arina · 300 kV · 25 mrad · 115 mm")
```